# 10 — Adaptive Window Reconstruction

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** use unstable windows from Notebook 09 to adapt reconstruction weights locally.

Notebook chain:

```text
07: global density-guided reconstruction
08: local gap-guided reconstruction
09: windowed reconstruction stability
10: adaptive window reconstruction
```

Notebook 09 showed:

> global reconstruction can look successful while local windows expose density, gap, and stability failures.

Notebook 10 upgrades fixed scoring into feedback scoring:

\[
w_{\mathrm{gap}}(W)=w_{\mathrm{gap}}\cdot(1+\lambda I(W))
\]

\[
w_{\mathrm{density}}(W)=\frac{w_{\mathrm{density}}}{1+\lambda I(W)}
\]

Core claim:

> Adaptive window reconstruction treats local instability as feedback, converting reconstruction from a fixed global rule into a local correction process.

## 0. Setup

Artifact structure:

```text
10_adaptive_window_reconstruction/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
10_adaptive_window_reconstruction_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "10_adaptive_window_reconstruction"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Adaptive Window Reconstruction"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Notebook 08 used fixed reconstruction weights.

Notebook 09 showed where fixed rules still have local instability.

Notebook 10 uses instability as feedback.

For a local window \(W=[a,b]\):

\[
I(W)=\alpha d(W)+\beta g(W)
\]

where:

- \(d(W)\) is local density error
- \(g(W)\) is local gap drift

Adaptive reconstruction changes candidate scoring inside unstable windows.

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

SCENARIOS = [
    {"name": "mixed_keep_75_noise_25", "kind": "mixed", "keep_fraction": 0.75, "noise_fraction": 0.25},
    {"name": "mixed_keep_50_noise_50", "kind": "mixed", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "mixed_keep_25_noise_100", "kind": "mixed", "keep_fraction": 0.25, "noise_fraction": 1.00},
    {"name": "biased_low_missing_high_noise", "kind": "biased_range", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_50", "kind": "adversarial_mod6", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_100", "kind": "adversarial_mod6", "keep_fraction": 0.25, "noise_fraction": 1.00},
]

Q_FILTER = int(math.sqrt(N_MAX))
COMPLETION_BIN_COUNT = 32

# Fixed local-gap weights from Notebook 08/09.
BASE_W_DENSITY = 0.45
BASE_W_GAP = 0.40
BASE_W_NEIGHBOR = 0.15

# Adaptive parameters.
LAMBDA_ADAPT = 1.75
INSTABILITY_ALPHA_DENSITY = 0.45
INSTABILITY_BETA_GAP = 0.55
MAX_GAP_WEIGHT_MULTIPLIER = 3.0
MIN_DENSITY_WEIGHT_MULTIPLIER = 0.25

# Sliding windows.
WINDOW_WIDTH = 10_000
WINDOW_STEP = 5_000
WINDOW_EDGES = [(a, min(a + WINDOW_WIDTH, N_MAX)) for a in range(2, N_MAX, WINDOW_STEP) if a + 100 <= N_MAX]

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "SCENARIOS": SCENARIOS,
    "Q_FILTER": Q_FILTER,
    "COMPLETION_BIN_COUNT": COMPLETION_BIN_COUNT,
    "BASE_W_DENSITY": BASE_W_DENSITY,
    "BASE_W_GAP": BASE_W_GAP,
    "BASE_W_NEIGHBOR": BASE_W_NEIGHBOR,
    "LAMBDA_ADAPT": LAMBDA_ADAPT,
    "INSTABILITY_ALPHA_DENSITY": INSTABILITY_ALPHA_DENSITY,
    "INSTABILITY_BETA_GAP": INSTABILITY_BETA_GAP,
    "MAX_GAP_WEIGHT_MULTIPLIER": MAX_GAP_WEIGHT_MULTIPLIER,
    "MIN_DENSITY_WEIGHT_MULTIPLIER": MIN_DENSITY_WEIGHT_MULTIPLIER,
    "WINDOW_WIDTH": WINDOW_WIDTH,
    "WINDOW_STEP": WINDOW_STEP,
    "WINDOW_COUNT": len(WINDOW_EDGES),
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 2. Reference primes and stress-test observations

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
prime_set = set(reference_primes.tolist())
universe = np.arange(2, N_MAX + 1)
composites = np.array([n for n in universe if n not in prime_set], dtype=int)
mod6_composites = composites[np.isin(composites % 6, [1, 5])]

def make_observation(cfg: dict) -> dict:
    name = cfg["name"]
    kind = cfg["kind"]
    keep_fraction = cfg["keep_fraction"]
    noise_fraction = cfg["noise_fraction"]

    keep_count = int(round(keep_fraction * len(reference_primes)))
    noise_count = int(round(noise_fraction * len(reference_primes)))

    if kind == "biased_range":
        midpoint = N_MAX // 2
        low_primes = reference_primes[reference_primes <= midpoint]
        high_primes = reference_primes[reference_primes > midpoint]

        low_keep_count = min(len(low_primes), int(round(0.90 * len(low_primes))))
        remaining_keep = max(0, keep_count - low_keep_count)
        high_keep_count = min(len(high_primes), remaining_keep)

        kept_low = rng.choice(low_primes, size=low_keep_count, replace=False)
        kept_high = rng.choice(high_primes, size=high_keep_count, replace=False) if high_keep_count else np.array([], dtype=int)

        high_composites = composites[composites > midpoint]
        noise_values = rng.choice(high_composites, size=min(noise_count, len(high_composites)), replace=False)

        values = np.sort(np.unique(np.concatenate([kept_low, kept_high, noise_values])))

    elif kind == "adversarial_mod6":
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(mod6_composites, size=min(noise_count, len(mod6_composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    else:
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(composites, size=min(noise_count, len(composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    return {
        "name": name,
        "kind": kind,
        "keep_fraction": keep_fraction,
        "noise_fraction": noise_fraction,
        "values": values,
    }

observations = [make_observation(cfg) for cfg in SCENARIOS]

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(reference_primes)),
    "composite_count": int(len(composites)),
    "mod6_composite_count": int(len(mod6_composites)),
    "scenario_count": int(len(observations)),
    "window_count": int(len(WINDOW_EDGES)),
}

summary

## 3. Filters and metric utilities

In [ ]:
def residue_filter(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = (values == 2) | (values == 3) | np.isin(values % 6, [1, 5])
    return np.sort(values[keep])

def passes_sieve(values: np.ndarray, q_max: int) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = np.ones(len(values), dtype=bool)
    filter_primes = reference_primes[reference_primes <= q_max]
    for q in filter_primes:
        keep &= ((values == q) | (values % q != 0))
    return keep

def sieve_filter(values: np.ndarray, q_max: int) -> np.ndarray:
    return np.sort(values[passes_sieve(values, q_max)])

def pi_model(x: np.ndarray | float) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    safe = np.maximum(x, 3.0)
    denom = np.log(safe) - 1.0
    denom = np.maximum(denom, 1.0)
    return safe / denom

def candidate_pool(q_max: int) -> np.ndarray:
    candidates = residue_filter(universe)
    return sieve_filter(candidates, q_max=q_max)

POOL = candidate_pool(Q_FILTER)

def values_in_window(values: np.ndarray, a: int, b: int) -> np.ndarray:
    values = np.sort(values)
    i = np.searchsorted(values, a, side="left")
    j = np.searchsorted(values, b, side="right")
    return values[i:j]

def window_gap_drift(values_window: np.ndarray) -> float:
    vals = np.sort(np.unique(values_window.astype(int)))
    if len(vals) < 3:
        return float("nan")
    gaps = np.diff(vals)
    anchors = vals[:-1]
    expected = np.maximum(np.log(np.maximum(anchors, 3)), 1.0)
    rel = np.abs(gaps - expected) / expected
    return float(np.mean(np.clip(rel, 0, 20)))

def window_density_error(values: np.ndarray, a: int, b: int) -> float:
    v_win = values_in_window(values, a, b)
    p_win = values_in_window(reference_primes, a, b)
    true_count = len(p_win)
    return abs(len(v_win) - true_count) / true_count if true_count else float("nan")

def window_metrics(values: np.ndarray, scenario: str, kind: str, method: str, a: int, b: int) -> dict:
    v_win = values_in_window(values, a, b)
    p_win = values_in_window(reference_primes, a, b)

    v_set = set(v_win.tolist())
    p_set = set(p_win.tolist())

    tp = len(v_set & p_set)
    fp = len(v_set - p_set)
    fn = len(p_set - v_set)

    true_count = len(p_win)
    obs_count = len(v_win)

    density_error = abs(obs_count - true_count) / true_count if true_count else float("nan")
    precision = tp / obs_count if obs_count else float("nan")
    recovery = tp / true_count if true_count else float("nan")
    f1 = 2 * precision * recovery / (precision + recovery) if (precision + recovery) else float("nan")
    gap_drift = window_gap_drift(v_win)

    return {
        "scenario": scenario,
        "kind": kind,
        "method": method,
        "window_left": int(a),
        "window_right": int(b),
        "window_mid": float(0.5 * (a + b)),
        "true_prime_count": int(true_count),
        "observed_count": int(obs_count),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "density_error": float(density_error),
        "precision": float(precision),
        "recovery": float(recovery),
        "f1": float(f1),
        "gap_drift": float(gap_drift),
    }

def nearest_gap_scores(candidates: np.ndarray, current_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    candidates = np.asarray(candidates, dtype=int)
    current_values = np.sort(np.unique(current_values.astype(int)))

    if len(current_values) == 0:
        return np.ones(len(candidates)), np.ones(len(candidates))

    pos = np.searchsorted(current_values, candidates)
    left_neighbor = np.where(pos > 0, current_values[np.maximum(pos - 1, 0)], -10**12)
    right_neighbor = np.where(pos < len(current_values), current_values[np.minimum(pos, len(current_values)-1)], 10**12)

    left_gap = candidates - left_neighbor
    right_gap = right_neighbor - candidates
    expected = np.maximum(np.log(np.maximum(candidates, 3)), 1.0)

    finite_left = left_neighbor > 0
    finite_right = right_neighbor < 10**11

    left_score = np.where(finite_left, np.exp(-np.abs(left_gap - expected) / expected), 0.5)
    right_score = np.where(finite_right, np.exp(-np.abs(right_gap - expected) / expected), 0.5)
    gap_score = 0.5 * (left_score + right_score)

    nearest = np.minimum(np.where(finite_left, left_gap, expected), np.where(finite_right, right_gap, expected))
    neighbor_score = 1.0 - np.exp(-nearest / expected)

    return gap_score, neighbor_score

print("Utilities ready")

## 4. Baseline reconstruction methods

In [ ]:
def complete_with_weights(
    filtered_values: np.ndarray,
    target_count: int,
    method_name: str,
    adaptive_weight_lookup: dict | None = None,
) -> tuple[np.ndarray, pd.DataFrame]:

    current = np.sort(np.unique(filtered_values.astype(int)))
    existing = set(current.tolist())
    available = np.array([v for v in POOL if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(current)))
    if needed_total == 0 or len(available) == 0:
        return current, pd.DataFrame(columns=["candidate", "method", "bin_left", "bin_right", "score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((current >= left) & (current <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)
        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)
        density_score = 1.0 - np.abs(bin_available - center) / scale

        gap_score, neighbor_score = nearest_gap_scores(bin_available, current)

        if adaptive_weight_lookup is None:
            w_density = np.full(len(bin_available), BASE_W_DENSITY)
            w_gap = np.full(len(bin_available), BASE_W_GAP)
            w_neighbor = np.full(len(bin_available), BASE_W_NEIGHBOR)
            instability = np.zeros(len(bin_available))
        else:
            w_density = np.zeros(len(bin_available))
            w_gap = np.zeros(len(bin_available))
            w_neighbor = np.zeros(len(bin_available))
            instability = np.zeros(len(bin_available))

            for i, c in enumerate(bin_available):
                adaptive = adaptive_weight_lookup.get(int(c), None)
                if adaptive is None:
                    w_density[i] = BASE_W_DENSITY
                    w_gap[i] = BASE_W_GAP
                    w_neighbor[i] = BASE_W_NEIGHBOR
                    instability[i] = 0.0
                else:
                    w_density[i] = adaptive["w_density"]
                    w_gap[i] = adaptive["w_gap"]
                    w_neighbor[i] = adaptive["w_neighbor"]
                    instability[i] = adaptive["instability"]

        raw_score = (
            w_density * density_score
            + w_gap * gap_score
            + w_neighbor * neighbor_score
            + rng.normal(0, 1e-6, size=len(bin_available))
        )

        norm = np.maximum(w_density + w_gap + w_neighbor, 1e-12)
        score = raw_score / norm

        order = np.argsort(score)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]

        lookup = {
            int(v): (
                float(score[i]),
                float(density_score[i]),
                float(gap_score[i]),
                float(neighbor_score[i]),
                float(w_density[i]),
                float(w_gap[i]),
                float(w_neighbor[i]),
                float(instability[i]),
            )
            for i, v in enumerate(bin_available)
        }

        for c in chosen:
            s, ds, gs, ns, wd, wg, wn, inst = lookup[int(c)]
            rows.append({
                "candidate": int(c),
                "method": method_name,
                "bin_left": int(left),
                "bin_right": int(right),
                "score": s,
                "density_score": ds,
                "gap_score": gs,
                "neighbor_score": ns,
                "w_density": wd,
                "w_gap": wg,
                "w_neighbor": wn,
                "instability": inst,
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("score", ascending=False).head(needed_total)
        selected = df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([current, selected])))
    return reconstructed, df

def density_only_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    # Density-only means set gap/neighbor weights to zero through a local shim.
    global BASE_W_DENSITY, BASE_W_GAP, BASE_W_NEIGHBOR
    old = (BASE_W_DENSITY, BASE_W_GAP, BASE_W_NEIGHBOR)
    BASE_W_DENSITY, BASE_W_GAP, BASE_W_NEIGHBOR = 1.0, 0.0, 0.0
    out = complete_with_weights(filtered_values, target_count, "density_only", None)
    BASE_W_DENSITY, BASE_W_GAP, BASE_W_NEIGHBOR = old
    return out

def fixed_local_gap_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    return complete_with_weights(filtered_values, target_count, "local_gap_fixed", None)

print("Baseline completion methods ready")

## 5. Window instability and adaptive weights

First pass:

1. build fixed local-gap reconstruction
2. measure local instability
3. assign adaptive weights by window
4. rebuild reconstruction using adaptive weights

In [ ]:
def compute_window_instability(values: np.ndarray, scenario: str, kind: str, method: str) -> pd.DataFrame:
    rows = []
    for a, b in WINDOW_EDGES:
        wm = window_metrics(values, scenario, kind, method, a, b)
        density_error = wm["density_error"]
        gap_drift = wm["gap_drift"]
        if np.isnan(gap_drift):
            gap_drift = 0.0
        instability = INSTABILITY_ALPHA_DENSITY * density_error + INSTABILITY_BETA_GAP * gap_drift
        wm["instability"] = float(instability)
        rows.append(wm)
    return pd.DataFrame(rows)

def build_adaptive_weight_lookup(window_instability_df: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []
    lookup = {}

    for _, row in window_instability_df.iterrows():
        a = int(row["window_left"])
        b = int(row["window_right"])
        instability = float(row["instability"])

        gap_multiplier = min(MAX_GAP_WEIGHT_MULTIPLIER, 1.0 + LAMBDA_ADAPT * instability)
        density_multiplier = max(MIN_DENSITY_WEIGHT_MULTIPLIER, 1.0 / (1.0 + LAMBDA_ADAPT * instability))

        w_density = BASE_W_DENSITY * density_multiplier
        w_gap = BASE_W_GAP * gap_multiplier
        w_neighbor = BASE_W_NEIGHBOR

        # Normalize to preserve score scale.
        total = w_density + w_gap + w_neighbor
        w_density_norm = w_density / total
        w_gap_norm = w_gap / total
        w_neighbor_norm = w_neighbor / total

        rows.append({
            "window_left": a,
            "window_right": b,
            "window_mid": float(row["window_mid"]),
            "density_error": float(row["density_error"]),
            "gap_drift": float(row["gap_drift"]),
            "instability": instability,
            "gap_multiplier": gap_multiplier,
            "density_multiplier": density_multiplier,
            "w_density": w_density_norm,
            "w_gap": w_gap_norm,
            "w_neighbor": w_neighbor_norm,
        })

    weights_df = pd.DataFrame(rows)

    # Candidate receives average adaptive weights from overlapping windows containing it.
    for candidate in POOL:
        c = int(candidate)
        sub = weights_df[(weights_df["window_left"] <= c) & (weights_df["window_right"] >= c)]
        if len(sub) == 0:
            continue
        lookup[c] = {
            "w_density": float(sub["w_density"].mean()),
            "w_gap": float(sub["w_gap"].mean()),
            "w_neighbor": float(sub["w_neighbor"].mean()),
            "instability": float(sub["instability"].mean()),
        }

    return lookup, weights_df

print("Adaptive utilities ready")

## 6. Run adaptive reconstruction comparison

Methods:

```text
raw
sieve_cleaned
density_only
local_gap_fixed
adaptive_window
```

In [ ]:
target_count = len(reference_primes)

reconstructed_sets = {}
candidate_completion_frames = []
adaptive_weight_frames = []

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]

    raw = np.sort(obs["values"])
    cleaned = sieve_filter(residue_filter(raw), q_max=Q_FILTER)

    density_recon, density_completions = density_only_completion(cleaned, target_count)
    fixed_recon, fixed_completions = fixed_local_gap_completion(cleaned, target_count)

    instability_df = compute_window_instability(fixed_recon, scenario, kind, "local_gap_fixed")
    adaptive_lookup, adaptive_weights_df = build_adaptive_weight_lookup(instability_df)

    adaptive_recon, adaptive_completions = complete_with_weights(
        cleaned,
        target_count,
        method_name="adaptive_window",
        adaptive_weight_lookup=adaptive_lookup,
    )

    reconstructed_sets[(scenario, "raw")] = raw
    reconstructed_sets[(scenario, "sieve_cleaned")] = cleaned
    reconstructed_sets[(scenario, "density_only")] = density_recon
    reconstructed_sets[(scenario, "local_gap_fixed")] = fixed_recon
    reconstructed_sets[(scenario, "adaptive_window")] = adaptive_recon

    for df in [density_completions, fixed_completions, adaptive_completions]:
        df["scenario"] = scenario
        df["kind"] = kind
        candidate_completion_frames.append(df)

    adaptive_weights_df["scenario"] = scenario
    adaptive_weights_df["kind"] = kind
    adaptive_weight_frames.append(adaptive_weights_df)

candidate_completions_df = pd.concat(candidate_completion_frames, ignore_index=True) if candidate_completion_frames else pd.DataFrame()
adaptive_weights_df = pd.concat(adaptive_weight_frames, ignore_index=True) if adaptive_weight_frames else pd.DataFrame()

[(k, len(v)) for k, v in list(reconstructed_sets.items())[:10]]

## 7. Window metrics and summaries

In [ ]:
method_order = ["raw", "sieve_cleaned", "density_only", "local_gap_fixed", "adaptive_window"]

window_rows = []
for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]

    for method in method_order:
        values = reconstructed_sets[(scenario, method)]
        for a, b in WINDOW_EDGES:
            window_rows.append(window_metrics(values, scenario, kind, method, a, b))

window_metrics_df = pd.DataFrame(window_rows)

summary_rows = []
for (scenario, kind, method), sub in window_metrics_df.groupby(["scenario", "kind", "method"]):
    summary_rows.append({
        "scenario": scenario,
        "kind": kind,
        "method": method,
        "mean_density_error": float(sub["density_error"].mean()),
        "max_density_error": float(sub["density_error"].max()),
        "mean_gap_drift": float(sub["gap_drift"].mean(skipna=True)),
        "max_gap_drift": float(sub["gap_drift"].max(skipna=True)),
        "mean_precision": float(sub["precision"].mean(skipna=True)),
        "mean_recovery": float(sub["recovery"].mean(skipna=True)),
        "mean_f1": float(sub["f1"].mean(skipna=True)),
        "unstable_density_windows_10pct": int((sub["density_error"] > 0.10).sum()),
        "unstable_gap_windows_gt1": int((sub["gap_drift"] > 1.0).sum()),
    })

method_summary_df = pd.DataFrame(summary_rows)

delta_rows = []
for scenario in [cfg["name"] for cfg in SCENARIOS]:
    density = method_summary_df[(method_summary_df["scenario"] == scenario) & (method_summary_df["method"] == "density_only")].iloc[0]
    fixed = method_summary_df[(method_summary_df["scenario"] == scenario) & (method_summary_df["method"] == "local_gap_fixed")].iloc[0]
    adaptive = method_summary_df[(method_summary_df["scenario"] == scenario) & (method_summary_df["method"] == "adaptive_window")].iloc[0]

    delta_rows.append({
        "scenario": scenario,
        "kind": adaptive["kind"],
        "adaptive_minus_density_mean_density_error": float(adaptive["mean_density_error"] - density["mean_density_error"]),
        "adaptive_minus_density_mean_gap_drift": float(adaptive["mean_gap_drift"] - density["mean_gap_drift"]),
        "adaptive_minus_fixed_mean_density_error": float(adaptive["mean_density_error"] - fixed["mean_density_error"]),
        "adaptive_minus_fixed_mean_gap_drift": float(adaptive["mean_gap_drift"] - fixed["mean_gap_drift"]),
        "adaptive_minus_fixed_unstable_gap_windows": int(adaptive["unstable_gap_windows_gt1"] - fixed["unstable_gap_windows_gt1"]),
        "adaptive_minus_fixed_unstable_density_windows": int(adaptive["unstable_density_windows_10pct"] - fixed["unstable_density_windows_10pct"]),
    })

method_deltas_df = pd.DataFrame(delta_rows)

measurement = {
    "mean_density_error_density_only": float(method_summary_df[method_summary_df["method"] == "density_only"]["mean_density_error"].mean()),
    "mean_density_error_local_gap_fixed": float(method_summary_df[method_summary_df["method"] == "local_gap_fixed"]["mean_density_error"].mean()),
    "mean_density_error_adaptive_window": float(method_summary_df[method_summary_df["method"] == "adaptive_window"]["mean_density_error"].mean()),
    "mean_gap_drift_density_only": float(method_summary_df[method_summary_df["method"] == "density_only"]["mean_gap_drift"].mean()),
    "mean_gap_drift_local_gap_fixed": float(method_summary_df[method_summary_df["method"] == "local_gap_fixed"]["mean_gap_drift"].mean()),
    "mean_gap_drift_adaptive_window": float(method_summary_df[method_summary_df["method"] == "adaptive_window"]["mean_gap_drift"].mean()),
    "mean_delta_adaptive_minus_fixed_gap_drift": float(method_deltas_df["adaptive_minus_fixed_mean_gap_drift"].mean()),
    "mean_delta_adaptive_minus_density_gap_drift": float(method_deltas_df["adaptive_minus_density_mean_gap_drift"].mean()),
    "mean_adaptive_gap_weight": float(adaptive_weights_df["w_gap"].mean()),
    "mean_adaptive_density_weight": float(adaptive_weights_df["w_density"].mean()),
}

cgcs = {
    "score": 1.0 / (
        1.0
        + measurement["mean_density_error_adaptive_window"]
        + measurement["mean_gap_drift_adaptive_window"]
    ),
    "definition": "adaptive stability score = 1/(1 + mean density error + mean gap drift) for adaptive window reconstruction",
    "interpretation": "Higher score means better adaptive local density and gap stability.",
}

measurement, cgcs, method_deltas_df

## 8. Figure 1 — adaptive weights by window

Adaptive feedback changes the gap and density weights by local instability.

In [ ]:
representative = "adversarial_mod6_noise_100"

fig, ax = plt.subplots(figsize=(12, 6))

sub = adaptive_weights_df[adaptive_weights_df["scenario"] == representative].sort_values("window_mid")

ax.plot(sub["window_mid"], sub["w_density"], marker="o", label="adaptive density weight")
ax.plot(sub["window_mid"], sub["w_gap"], marker="o", label="adaptive gap weight")
ax.plot(sub["window_mid"], sub["w_neighbor"], marker="o", label="adaptive neighbor weight")
ax.set_title(f"Adaptive weights by window — {representative}")
ax.set_xlabel("window midpoint")
ax.set_ylabel("normalized weight")
ax.legend()
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_adaptive_weights_by_window.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 9. Figure 2 — window instability profile

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(sub["window_mid"], sub["density_error"], marker="o", label="density error")
ax.plot(sub["window_mid"], sub["gap_drift"], marker="o", label="gap drift")
ax.plot(sub["window_mid"], sub["instability"], marker="o", label="instability")
ax.set_title(f"Window instability profile — {representative}")
ax.set_xlabel("window midpoint")
ax.set_ylabel("score")
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_window_instability_profile.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 10. Figure 3 — method density error

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    s = method_summary_df[method_summary_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, s["mean_density_error"], marker="o", label=scenario)

ax.set_title("Mean window density error by method")
ax.set_xlabel("method")
ax.set_ylabel("mean local density error")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_method_density_error.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 11. Figure 4 — method gap drift

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    s = method_summary_df[method_summary_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, s["mean_gap_drift"], marker="o", label=scenario)

ax.set_title("Mean window gap drift by method")
ax.set_xlabel("method")
ax.set_ylabel("mean local gap drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_method_gap_drift.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

## 12. Figure 5 — adaptive gap-drift delta

Negative values mean adaptive reconstruction improved over fixed local-gap reconstruction.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
labels = method_deltas_df["scenario"].str.replace("_", "\n")

ax.bar(labels, method_deltas_df["adaptive_minus_fixed_mean_gap_drift"])
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Adaptive minus fixed local-gap mean window gap drift")
ax.set_xlabel("scenario")
ax.set_ylabel("delta mean gap drift")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_adaptive_delta_gap_drift.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

## 13. Figure 6 — unstable windows comparison

In [ ]:
methods_compare = ["density_only", "local_gap_fixed", "adaptive_window"]
unstable_gap = method_summary_df.pivot(index="scenario", columns="method", values="unstable_gap_windows_gt1").loc[[cfg["name"] for cfg in SCENARIOS], methods_compare]
unstable_density = method_summary_df.pivot(index="scenario", columns="method", values="unstable_density_windows_10pct").loc[[cfg["name"] for cfg in SCENARIOS], methods_compare]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(unstable_gap.index))
width = 0.25

for i, method in enumerate(methods_compare):
    ax.bar(x + (i - 1) * width, unstable_gap[method], width, label=method)

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", "\n") for s in unstable_gap.index], rotation=45, ha="right")
ax.set_title("Unstable gap windows comparison")
ax.set_xlabel("scenario")
ax.set_ylabel("count of windows with gap drift > 1")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_unstable_windows_comparison.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

## 14. Figure 7 — gap profile under adaptive reconstruction

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for method in ["density_only", "local_gap_fixed", "adaptive_window"]:
    s = window_metrics_df[
        (window_metrics_df["scenario"] == representative)
        & (window_metrics_df["method"] == method)
    ].sort_values("window_mid")
    ax.plot(s["window_mid"], s["gap_drift"], marker="o", markersize=3, label=method)

ax.set_title(f"Gap drift profile — {representative}")
ax.set_xlabel("window midpoint")
ax.set_ylabel("window gap drift")
ax.legend()
ax.grid(True, alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_profile_adversarial.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

## 15. Figure 8 — adaptive stability map

In [ ]:
heat_rows = []

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    for method in ["density_only", "local_gap_fixed", "adaptive_window"]:
        s = window_metrics_df[
            (window_metrics_df["scenario"] == scenario)
            & (window_metrics_df["method"] == method)
        ].sort_values("window_mid")
        for _, row in s.iterrows():
            heat_rows.append({
                "scenario_method": f"{scenario} | {method}",
                "window_mid": row["window_mid"],
                "gap_drift": row["gap_drift"],
            })

heat_df = pd.DataFrame(heat_rows)
pivot = heat_df.pivot(index="scenario_method", columns="window_mid", values="gap_drift")

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot.values, aspect="auto", interpolation="nearest")
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xticks(np.linspace(0, len(pivot.columns)-1, 8).astype(int))
ax.set_xticklabels([int(pivot.columns[i]) for i in np.linspace(0, len(pivot.columns)-1, 8).astype(int)], rotation=45)
ax.set_title("Adaptive windowed gap drift stability map")
ax.set_xlabel("window midpoint")
ax.set_ylabel("scenario | method")
fig.colorbar(im, ax=ax, label="gap drift")

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_stability_map_adaptive.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

## 16. Figure 9 — F1 by method

Classification metrics can still saturate, so use this as a secondary diagnostic.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    s = method_summary_df[method_summary_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, s["mean_f1"], marker="o", label=scenario)

ax.set_title("Mean window F1 by method")
ax.set_xlabel("method")
ax.set_ylabel("mean local F1")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig9_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_window_f1_by_method.png"
fig.savefig(fig9_path, dpi=180, bbox_inches="tight")
plt.show()

fig9_path

## 17. Interpretation

Adaptive reconstruction converts local instability into local scoring feedback.

Core result to test:

> Adaptive reconstruction should reduce localized gap drift relative to fixed local-gap scoring while preserving near-zero density error.

If improvement is small, that means fixed density/local-gap reconstruction has already saturated this model class and the next improvement requires a better expected-gap model.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook tests adaptive window reconstruction.",
    "",
    "Notebook 09 showed that local-window diagnostics expose failures hidden by global metrics.",
    "",
    "Notebook 10 uses local instability as feedback for candidate scoring.",
    "",
    "## Adaptive rule",
    "",
    "For a window W:",
    "",
    "```text",
    "instability(W) = alpha * density_error(W) + beta * gap_drift(W)",
    "w_gap(W) = w_gap * (1 + lambda * instability(W))",
    "w_density(W) = w_density / (1 + lambda * instability(W))",
    "```",
    "",
    "## Summary metrics",
    "",
    f"- mean density error, density-only = {measurement['mean_density_error_density_only']:.6f}",
    f"- mean density error, fixed local-gap = {measurement['mean_density_error_local_gap_fixed']:.6f}",
    f"- mean density error, adaptive window = {measurement['mean_density_error_adaptive_window']:.6f}",
    f"- mean gap drift, density-only = {measurement['mean_gap_drift_density_only']:.6f}",
    f"- mean gap drift, fixed local-gap = {measurement['mean_gap_drift_local_gap_fixed']:.6f}",
    f"- mean gap drift, adaptive window = {measurement['mean_gap_drift_adaptive_window']:.6f}",
    f"- adaptive minus fixed gap drift = {measurement['mean_delta_adaptive_minus_fixed_gap_drift']:.6f}",
    f"- adaptive stability CGCS score = {cgcs['score']:.6f}",
    "",
    "## Interpretation",
    "",
    "Adaptive reconstruction uses local instability as feedback.",
    "",
    "If adaptive reconstruction improves gap drift, local feedback adds value beyond fixed scoring.",
    "",
    "If improvement is small, fixed density/local-gap reconstruction has saturated the log-gap model.",
    "",
    "## Core statement",
    "",
    "Adaptive window reconstruction treats local instability as feedback, converting reconstruction from a fixed global rule into a local correction process.",
    "",
    "## Next limit",
    "",
    "If adaptive feedback saturates, the next notebook should replace log(x) with a learned local expected-gap model.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path, fig7_path, fig8_path, fig9_path]
figure_titles = [
    "Adaptive weights by window",
    "Window instability profile",
    "Method density error",
    "Method gap drift",
    "Adaptive gap-drift delta",
    "Unstable windows comparison",
    "Gap profile under adaptive reconstruction",
    "Adaptive stability map",
    "Mean window F1 by method",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 18. Export data, docs, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
window_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_metrics.csv"
method_summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_method_summary.csv"
adaptive_weights_path = DATA_DIR / f"{NOTEBOOK_NUM}_adaptive_weights.csv"
method_deltas_path = DATA_DIR / f"{NOTEBOOK_NUM}_method_deltas.csv"
candidate_completions_path = DATA_DIR / f"{NOTEBOOK_NUM}_candidate_completions.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
window_metrics_df.to_csv(window_metrics_path, index=False)
method_summary_df.to_csv(method_summary_path, index=False)
adaptive_weights_df.to_csv(adaptive_weights_path, index=False)
method_deltas_df.to_csv(method_deltas_path, index=False)
candidate_completions_df.to_csv(candidate_completions_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "window_metrics": str(window_metrics_path),
        "method_summary": str(method_summary_path),
        "adaptive_weights": str(adaptive_weights_path),
        "method_deltas": str(method_deltas_path),
        "candidate_completions": str(candidate_completions_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 10 converts window instability from Notebook 09 into adaptive reconstruction weights.",
    "",
    "## Methods compared",
    "",
    "1. raw",
    "2. sieve_cleaned",
    "3. density_only",
    "4. local_gap_fixed",
    "5. adaptive_window",
    "",
    "## Adaptive rule",
    "",
    "Instability increases gap weight and decreases density weight locally.",
    "",
    "## Diagnostics",
    "",
    "1. adaptive weights by window",
    "2. instability profile",
    "3. mean local density error",
    "4. mean local gap drift",
    "5. adaptive delta against fixed local-gap",
    "6. unstable-window counts",
    "7. stability heatmap",
    "",
    "## Core claim",
    "",
    "Adaptive window reconstruction treats local instability as feedback.",
    "",
    "## Handoff",
    "",
    "Notebook 11 should replace log(x) with a locally learned expected-gap model.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook uses local instability as reconstruction feedback.",
    "",
    r"For a window $W$, define",
    r"\[",
    r"I(W)=\alpha d(W)+\beta g(W).",
    r"\]",
    "",
    r"Adaptive weights are",
    r"\[",
    r"w_{\mathrm{gap}}(W)=w_{\mathrm{gap}}\left(1+\lambda I(W)\right),",
    r"\]",
    r"\[",
    r"w_{\mathrm{density}}(W)=\frac{w_{\mathrm{density}}}{1+\lambda I(W)}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item adaptive mean density error $= {measurement['mean_density_error_adaptive_window']:.6f}$",
    rf"  \item adaptive mean gap drift $= {measurement['mean_gap_drift_adaptive_window']:.6f}$",
    rf"  \item adaptive minus fixed gap drift $= {measurement['mean_delta_adaptive_minus_fixed_gap_drift']:.6f}$",
    rf"  \item adaptive stability CGCS score $= {cgcs['score']:.6f}$",
    r"\end{itemize}",
    "",
    r"Adaptive window reconstruction treats local instability as feedback.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Adaptive Window Reconstruction}",
    "",
    r"\subsection*{Window instability}",
    r"\[",
    r"I(W)=\alpha d(W)+\beta g(W).",
    r"\]",
    "",
    r"\subsection*{Adaptive gap weight}",
    r"\[",
    r"w_{\mathrm{gap}}(W)=w_{\mathrm{gap}}\left(1+\lambda I(W)\right).",
    r"\]",
    "",
    r"\subsection*{Adaptive density weight}",
    r"\[",
    r"w_{\mathrm{density}}(W)=\frac{w_{\mathrm{density}}}{1+\lambda I(W)}.",
    r"\]",
    "",
    r"\subsection*{Adaptive candidate score}",
    r"\[",
    r"score(x,W)=",
    r"w_d(W)s_d(x)+w_g(W)s_g(x)+w_n s_n(x).",
    r"\]",
    "",
    r"\subsection*{Adaptive stability score}",
    r"\[",
    r"CGCS_{\mathrm{adaptive}}=",
    r"\frac{1}{1+\overline{d(W)}+\overline{g(W)}}.",
    r"\]",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, window_metrics_path, method_summary_path, adaptive_weights_path, method_deltas_path, candidate_completions_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 19. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 20. Next notebook handoff

Next notebook:

```text
11_learned_gap_expectation.ipynb
```

Purpose:

> replace \(\log(x)\) with a locally learned expected-gap model.

In [ ]:
next_step = "Notebook 11: learned gap expectation."
print(next_step)